In [1]:
# Autoreload
%load_ext autoreload
%autoreload 2

from optimizer import Optimizer
from chromosome import Chromosome
from character import Character
from utils import get_item_set, elements
import json

with open('data/MAPPED_ITEMS.json', 'r') as file:
    items = {i["ankama_id"]: i for i in json.load(file)}

with open('data/MAPPED_SETS.json', 'r') as file:
    item_sets = {i["ankama_id"]: i for i in json.load(file)}

In [2]:
config = {
            "optimizer" : {
                "language": "es",
                "population_size": 300,
                "num_parents_mating": 100, # Number of parents to mate, usually 1/3 of population size
                "num_generations": 400,
                "mutation_rate": 0.05,
                "crossover_rate": 0.8,
                "parent_selection_type": "tournament",  # Tournament Selection
                "tournament_size": 3,
                "exclusions": {
                    "items": [6894,
                              6895,
                              3080,
                              9031,
                              6980 # Vulbis
                              ]  # Excluded items
                },
                "inclusions": {
                    "items": [
                        14093, # Botas kuko
                        18718, # Escudo kuko
                        14092, # Anillo kuko
                        8993, # Espada maldita
                        7754, # Dofus ocre
                    ]
                },
                "preferences": {
                    "lower": {
                        12: {
                            "target": 12,  # AP
                            "strength": 0.9,  # Penalty for not meeting the target
                        },
                        8: {
                            "target": 6,  # MP
                            "strength": 0.9,  # Penalty for not meeting the target
                        },
                        29: {
                            "target": 80,  # Crit
                            "strength": 0.75,  # Penalty for not meeting the target
                        },
                        9: {
                            "target": 4000,  # Vit
                            "strength": 0.5,  # Penalty for not meeting the target
                        },
                        34 : {
                            "target": 15, # %res neutral
                            "strength": 0.1,  # Penalty for not meeting the target
                        },
                        37 : {
                            "target": 15, # %res fire
                            "strength": 0.1,  # Penalty for not meeting the target
                        },
                        16 : {
                            "target": 15, # %res air
                            "strength": 0.1,  # Penalty for not meeting the target
                        },
                        17 : {
                            "target": 15, # %res water
                            "strength": 0.1,  # Penalty for not meeting the target
                        },
                        63 : {
                            "target": 15, # %res range
                            "strength": 0.1,  # Penalty for not meeting the target
                        }
                    },
                    "upper": {}
                },

                "level_offset": 10,  # Level offset for items
                "objective": "weapon"  # Objective to optimize, can be "weapon" or "elements"
            },
            "character": {
                "level": 200,
                "scrolled": True,
                "exo": {
                    12:1, # Exo AP
                    8:1
                    },  # Exo MP
                "distributed_points": {
                    9: 995,  # Vitality
                },
                "elements" : [
                    36, # Agility,
                    22, # Chance
                    13, # Intelligence
                    45  # Strength
                ],
            },
        }

In [3]:
import pandas as pd

effect_descriptions = {}
for _,item in items.items():
    if not item["effects"]:
        continue
    for effect in item["effects"]:
        effect_descriptions[effect["element_id"]] = effect["type"]

dct = {}
for k,v in items.items():
    item_set = get_item_set(v,item_sets)
    dct[k] = {
        "name" : v["name"][config["optimizer"]["language"]],
        "set" : item_set["name"][config["optimizer"]["language"]] if item_set else None,
        "level" : v["level"]
    }
Items = pd.DataFrame.from_dict(dct, orient='index')

In [4]:
Items[Items["name"].str.contains("kuko",case=False)]

,name,set,level
13989,Pluma de kukoloide,None,200
14091,Amuleto de kukoloide,Set de kukoloide,200
14092,Anillo de kukoloide,Set de kukoloide,200
14093,Botas de kukoloide,Set de kukoloide,200
13988,Ojo de kukoloide,None,200
18718,Escudo de kukoloide,Set de kukoloide,200


In [5]:
char = Character(**config["character"])
opt = Optimizer(char,
                 items,
                   item_sets,
                     **config["optimizer"])

Total items: 2748
Total item sets: 148
Total pools:
   16 with sizes [69, 1, 1, 71, 62, 1, 1, 88, 69, 412, 1, 326, 326, 326, 326, 326]
Total combinations: 10^24.45


In [6]:
solutions = opt.optimize(islands = 10, max_workers = 4)

In [7]:
for idx, solution in enumerate(sorted(solutions, key=lambda x: x.fitness, reverse=True)):
    if solution.penalized:
        continue
    print(f"Solution {idx}:")
    print("\tFitness:", solution.get_fitness())
    print("\tWeapon damage:", solution.get_final_weapon_damage())
    print("\tElemental damage:", solution.get_final_elemental_damage())
    # print("\tIs the solution viable?", solution.viable)
    # print("\tWas the solution penalized?", solution.penalized)

    print("\tPreferences:")
    df = solution.totals_summary(effect_descriptions)
    for bound, preference in solution.preferences.items():
        for element, details in preference.items():
            row = df.loc[element]
            print(f"\t\t{row['description']} : {row['value']}")
    print("\tCharacteristics:")
    for element in elements:
        print(f"\t\t{df.loc[element]['description']}: {df.loc[element]['value']}")

    print("\tItems:")
    print("\t\tType\tDescription\tLevel")
    for item,row in solution.set_summary().sort_values("type").iterrows():
        # Prettify the output
        print(f"\t\t{row['type']}\t{row['description']}\t{row['level']}")
    print("\tChromosome:")
    print("\t\t", solution.chromosome)

Solution 0:
	Fitness: 1936.7600000000002
	Weapon damage: 1936.7600000000002
	Elemental damage: 1485.0
	Preferences:
		PA : 12.0
		PM : 6.0
		% crítico : 88.0
		vitalidad : 4560.0
		% resistencia neutral : 17.0
		% resistencia al fuego : 25.0
		% resistencia al aire : 44.0
		% resistencia al agua : 33.0
		% resistencia a la tierra : 33.0
	Characteristics:
		agilidad: 530.0
		suerte: 200.0
		inteligencia: 530.0
		fuerza: 480.0
		vitalidad: 4560.0
		sabiduría: 400.0
	Items:
		Type	Description	Level
		Amuleto	Amuleto de estrígido	200
		Anillo	Anillo de Noai Aludem	197
		Anillo	Anillo de kukoloide	200
		Bota	Botas de kukoloide	200
		Capa	Capa Pología	198
		Cinturón	Cinturón de estrígido	200
		Dofus	Dofus Púrpura	110
		Dofus	Dofus Turquesa	160
		Dofus	Dofus de los hielos	180
		Dofus	Dofus Marfil	180
		Dofus	Dofus Ocre	160
		Dofus	Dolmanax	100
		Escudo	Escudo de kukoloide	200
		Espada	Espada maldita del gran Desangrador Guerrero	200
		Mascota	Microsmoglob	20
		Sombrero	Gorro de Noai Aludem	19

In [8]:
solution.totals_summary(effect_descriptions)#.sort_values("value",ascending=False)

,description,value
0,Intercambiable:,0.0
8,PM,6.0
9,vitalidad,4410.0
10,sabiduría,410.0
12,PA,12.0
13,inteligencia,380.0
14,de resistencia al fuego,10.0
15,de resistencia a la tierra,25.0
16,% resistencia al aire,34.0
17,% resistencia al agua,20.0


In [9]:
set_summary = solution.set_summary()
set_summary

,type_id,type,description,level,set
694,13,Dofus,Dofus Púrpura,110,None
739,13,Dofus,Dofus Turquesa,160,None
7043,13,Dofus,Dofus de los hielos,180,None
7754,13,Dofus,Dofus Ocre,160,None
8993,2,Espada,Espada maldita del gran Desangrador Guerrero,200,None
12117,11,Capa,Capa de Fuji Gélifux,193,Set de Fuji Gélifux
12118,10,Sombrero,Tocado de Fuji Gélifux,195,Set de Fuji Gélifux
12120,3,Anillo,Anillo de Fuji Gélifux,192,Set de Fuji Gélifux
13344,13,Dofus,Dolmanax,100,None
13673,12,Mascota,Lokulto,20,None


In [10]:
set_summary["set"].value_counts()

set
Set de Fuji Gélifux    3
Set de kukoloide       3
Set de estrígido       2
Name: count, dtype: int64

In [11]:
import matplotlib.pyplot as plt
plt.plot(opt.ga_instance.best_solutions_fitness)
plt.xlabel('Generation')
plt.ylabel('Best Fitness')

AttributeError: 'Optimizer' object has no attribute 'ga_instance'

In [ ]:
assert False

AssertionError: 

In [14]:
# Current
chr = Chromosome(
                 [
                    8993, # Espada
                    13641, 13642, # Noai
                    14161, 14162, 14169, # Charlatan
                    14092, 14093, 18718, # Kuko
                    13673, # Lokulto
                    739, 694, 7754, 737, 7043, 17078 # Dofus
                 ],
                 opt)
chr.fitness

2049.74

In [ ]:
Items[Items["name"].str.contains("kuko")]

,name,set,level
13989,Pluma de kukoloide,None,200
14091,Amuleto de kukoloide,Set de kukoloide,200
14092,Anillo de kukoloide,Set de kukoloide,200
14093,Botas de kukoloide,Set de kukoloide,200
13988,Ojo de kukoloide,None,200
18718,Escudo de kukoloide,Set de kukoloide,200


In [ ]:
chr.set_summary()

In [ ]:
chr.set_summary()["set"].value_counts()

In [ ]:
chr.totals_summary(effect_descriptions)

In [ ]:
Items[Items["name"].str.contains("Chani")]

In [ ]:
item = items[21392]
# item = items[6981]

In [ ]:
chr.are_item_conditions_met(item)

In [ ]:
item

In [ ]:
Items

In [ ]:
lst

In [ ]:
lst[0][2]

In [ ]:
chr.are_item_conditions_met(items[11738])

In [ ]:
items[11738]["conditions"]

In [ ]:
chr.totals[8]

In [ ]:
lst = []
for i in items.values():
    if i["type"]["superTypeId"] not in [i[0] for i in opt.pool_types]:
        continue
    if i["conditions"]:
        print(opt.solution.are_item_conditions_met(i))
        lst.append((i,opt.solution.are_item_conditions_met(i)))

In [ ]:
lst[1]

In [ ]:
lst.count(True), lst.count(False)